# Step 5: FP8 — E4M3 vs E5M2（推理用 E4M3）

**目标**：理解 FP8 两种格式（E4M3 / E5M2）的表示范围与精度，实现"cast + clamp"式 FP8 量化，并亲见 **E4M3 对正常值误差远小于 INT8**（因为浮点动态范围自带抗离群——s1 的 emergent outlier 不再击垮量化）。

**对应 OUTLINE 课时**：1.5 FP8（~45 分钟）。

> M2 用 llm-compressor 出 FP8 产物；M1 从零实现 FP8 的数值语义，理解 *为什么* FP8 对 LLM 友好。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理

### FP8 = 1 位符号 + 指数 + 尾数（共 8 位）

| 格式 | 符号 | 指数 E | 尾数 M | 最大正数 | 最小正常 | 特点 |
|------|------|--------|--------|----------|----------|------|
| **E4M3** | 1 | 4 | 3 | **448** | 2^-6≈0.0156 | 尾数多 → **精度高**，范围窄。**推理首选** |
| **E5M2** | 1 | 5 | 2 | **57344** | 2^-14≈6e-5 | 指数多 → **范围大**，精度低。训练梯度用 |

- **E4M3 max = 448**：因为指数 4 位（含 bias），最大有效指数受限；尾数 3 位给 3 个精度位。
- **E5M2 max = 57344**：指数 5 位范围更大，但尾数只有 2 位 → 精度粗。

### 为什么推理用 E4M3？

激活/权重的典型幅值分布在 $[-10, 10]$ 量级（远小于 448），E4M3 的范围足够，且**尾数多 = 精度高**。关键优势：**浮点动态范围**——同一个 E4M3 格式能同时表示 0.01 和 100（INT8 做不到，INT8 的分辨率是均匀的）。所以 emergent outlier（s1 的 20× channel）**不会击垮 E4M3 量化**：大值和小值各自被映射到合适的浮点码值，不像 INT8 那样被一个全局 scale 压扁。

### "cast + clamp" 量化

FP8 量化的最简形式（硬件原生路径就是 cast）：把 FP32/FP16 张量先 clamp 到 $[-448, 448]$，再 cast 成 `float8_e4m3fn`，再 cast 回 FP16 反量化。误差来自尾数截断（约 1/8 相对误差），但因为是浮点，相对误差随幅值自适应。

## 本步填空

1. **`fp8_max_value(format)`** —— 返回 E4M3=448 / E5M2=57344（判断型：记住两种格式的范围）。
2. **`quantize_to_fp8(x, fmt)`** —— cast+clamp 式 FP8 量化，返回反量化后的近似张量（第二参 `fmt` 是 str："e4m3"/"e5m2"）。

In [ ]:
def fp8_max_value(fmt="e4m3"):
    """返回指定 FP8 格式的最大可表示正数。

    参数
    ----
    fmt : str，"e4m3" 或 "e5m2"。

    返回
    ----
    int：e4m3 → 448；e5m2 → 57344。

    推导（指数位 → 动态范围，不是死记两个数；IEEE 754 P3109 最大有限值）
    ------------------------------------------------------------------
    FP8 = 1 符号 + E 指数 + M 尾数。最大正有限值 = (尾数全 1 的有效数字) × (最大有效
    指数对应的 2 的幂)。两个格式这里**最容易搞错**，关键是「指数全 1」怎么解释：

        有效数字（尾数全 1）= 1 + (2^M − 1)/2^M = 1.75        # E4M3、E5M2 都是 1.75

      • **E4M3（E=4, M=3，bias=7）—— 无 inf 表示**：
        指数域全 1（1111）仍合法 finite（E4M3 把"全 1 指数"留给最大有限值而非 inf）。
        max 指数 1111 = 15，unbiased = 15 − 7 = 8 → 2^8 = 256。
        max = 1.75 × 2^8 = 1.75 × 256 = **448**
      • **E5M2（E=5, M=2，bias=15）—— 有 inf 表示**：
        指数域全 1（11111）是 inf/nan，不能用于 finite。最大 finite 指数 = 11110 = 30，
        unbiased = 30 − 15 = 15 → 2^15 = 32768。
        max = 1.75 × 2^15 = 1.75 × 32768 = **57344**

    ⚠ 最易错点：两个格式**最大有效指数不同**——E4M3 没有-inf（全 1 指数仍是有限值 2^8），
    E5M2 有一-inf（全 1 指数是 inf，最大有限指数只能取 11110 即 2^15）。这就是为什么
    E5M2 范围比 E4M3 大约 128×（2^15 / 2^8 = 256，但有效数字略不同）。代价：E5M2 尾数
    少一位（M2 vs M3），精度更粗。→ **推理选 E4M3**（范围够 + 精度高），梯度用 E5M2（范围大）。
    """
    # TODO: 根据 fmt 返回 448 或 57344。
    raise NotImplementedError


def quantize_to_fp8(x, fmt="e4m3"):
    """cast + clamp 式 FP8 量化，返回反量化后的近似张量（float32）。

    参数
    ----
    x : torch.Tensor，任意形状，float。
    fmt : "e4m3" 或 "e5m2"。

    返回
    ----
    torch.Tensor，与 x 同形 float32：x → clamp 到 [-max, max] → cast 成 FP8 → cast 回 float32。
      - e4m3 对应 torch.float8_e4m3fn
      - e5m2 对应 torch.float8_e5m2

    提示
    ----
      - max = fp8_max_value(fmt)（你刚实现的）。
      - 选 dtype：e4m3 → torch.float8_e4m3fn；e5m2 → torch.float8_e5m2。
      - 三步：x_clamped = x.clamp(-max, max)；q = x_clamped.to(fp8_dtype)；return q.to(torch.float32)。
      - torch cu128 起 float8 dtype 在 CPU/GPU 上都能直接 .to() cast（本课程已验证）；
        不需要额外 fallback。下方脚手架 _fp8_e4m3_emulate 是"概念演示用"的纯算术模拟，
        本函数不用调它——直接走真 cast 即可（L1/L2/L3 三层都能跑）。
    """
    # TODO: 实现 cast+clamp FP8 量化。
    raise NotImplementedError


# 脚手架（提供）：纯算术的 E4M3 模拟（概念演示，展示"浮点尾数截断"的数值语义，本课主路径用真 cast）
def _fp8_e4m3_emulate(x):
    """用符号+指数+尾数分解手工模拟 E4M3 的尾数截断（与真 cast 量级一致，便于讲解浮点量化原理）。"""
    x = x.float().clamp(-448, 448)
    import math
    sign = x.sign()
    mag = x.abs().clamp_min(1e-20)
    # 把幅值量化到 E4M3 的 8 个尾数级别（每个 2 的幂上 8 级）
    exp = torch.floor(torch.log2(mag))
    frac = mag / (2.0 ** exp)            # in [1,2)
    frac_q = torch.round(frac * 8) / 8   # 8 级尾数
    return (sign * frac_q * (2.0 ** exp)).float()

In [ ]:
%%ipytest -qq

def test_fp8_max_value():
    assert fp8_max_value("e4m3") == 448
    assert fp8_max_value("e5m2") == 57344

def test_fp8_max_value_unknown_raises():
    try:
        fp8_max_value("e6m1"); assert False, "未知格式应报错"
    except (ValueError, KeyError):
        pass

def test_quantize_to_fp8_preserves_normal_values():
    x = torch.tensor([0.5, 1.0, -2.0, 3.5])
    q = quantize_to_fp8(x, "e4m3")
    assert q.shape == x.shape
    # 正常值 E4M3 误差应很小（相对误差 < 12.5% = 1/8 尾数）
    rel = ((q - x).abs() / x.abs().clamp_min(1e-9)).mean().item()
    assert rel < 0.13, f"E4M3 正常值相对误差应 <12.5%，实际 {rel:.4f}"

def test_quantize_to_fp8_clamps_large_values():
    x = torch.tensor([10000.0, -50000.0])
    q = quantize_to_fp8(x, "e4m3")
    # E4M3 max=448，超大值应被 clamp
    assert q[0].item() <= 448.0 + 1e-3
    assert q[1].item() >= -448.0 - 1e-3

def test_fp8_better_than_int8_on_outlier_mix():
    """判断型：含 outlier 的激活，E4M3 误差应 < INT8（浮点动态范围抗离群）。"""
    torch.manual_seed(0)
    x = torch.randn(256)
    x[10] *= 40.0; x[200] *= 30.0   # emergent outlier
    # E4M3 量化
    q_fp8 = quantize_to_fp8(x, "e4m3")
    err_fp8 = ((q_fp8 - x).abs().mean() / x.abs().mean()).item()
    # 对照：per-tensor INT8（一个全局 scale，被 outlier 拉爆）
    scale = 127.0 / x.abs().max().clamp_min(1e-9)
    q_int8 = torch.round(x * scale).clamp(-127, 127).to(torch.int8).float() / scale
    err_int8 = ((q_int8 - x).abs().mean() / x.abs().mean()).item()
    assert err_fp8 < err_int8, f"FP8 误差 {err_fp8:.4f} 应 < INT8 {err_int8:.4f}（抗离群）"

## L2：tiny 验证（CPU）—— E4M3 误差 << INT8

合成验证：E4M3 对正常值误差远小于 INT8 per-tensor。torch cu128 起 `float8_e4m3fn` dtype 在 **CPU 上也能直接 `.to()` cast**（本课程已验证），故 L2 直接走与 L1/L3 同一条真 cast 路径 `quantize_to_fp8(x, "e4m3")`——与 L1 测的、L3 跑的是同一函数。

In [ ]:
# E4M3 vs INT8 per-tensor 对比（CPU 模拟 E4M3）
torch.manual_seed(7)
x = torch.randn(512) * 2
x[5] *= 50.0; x[100] *= 35.0   # outlier

# INT8 per-tensor
si = 127.0 / x.abs().max()
q_int8 = torch.round(x*si).clamp(-127,127).to(torch.int8).float()/si
err_int8 = ((q_int8 - x).abs().mean()/x.abs().mean()).item()

# E4M3 真 cast（与 L1/L3 同一函数，CPU 可跑——cu128 起 float8 dtype 支持 CPU cast）
q_fp8 = quantize_to_fp8(x, "e4m3")
err_fp8 = ((q_fp8 - x).abs().mean()/x.abs().mean()).item()

print(f"INT8 per-tensor 相对误差: {err_int8:.4f}（被 outlier 拉爆）")
print(f"E4M3 真 cast 相对误差: {err_fp8:.4f}（浮点动态范围抗离群）")
assert err_fp8 < err_int8
print("L2a PASS：E4M3 误差 << INT8 per-tensor（emergent outlier 不击垮 FP8）")

# tiny Qwen2：FP8 max 语义（纯 CPU，只验 fp8_max_value + clamp 逻辑）
from transformers import Qwen2Config, Qwen2ForCausalLM
tiny = Qwen2ForCausalLM(Qwen2Config(num_hidden_layers=1, hidden_size=128,
    intermediate_size=256, num_attention_heads=4, num_key_value_heads=2,
    vocab_size=320, tie_word_embeddings=True)).eval()
w = tiny.model.layers[0].mlp.gate_proj.weight.data.float()
max4 = fp8_max_value("e4m3"); max5 = fp8_max_value("e5m2")
print(f"\nfp8_max: e4m3={max4}, e5m2={max5}")
print(f"tiny gate_proj 权重范围: [{w.min():.3f}, {w.max():.3f}]（远 < 448 → E4M3 范围足够）")
assert w.abs().max().item() < 448, "正常权重幅值应远小于 E4M3 max"
print("L2b PASS：FP8 范围语义正确，正常权重/激活 << 448")

## L3：H200 执行（真 0.5B gate_proj，torch.float8_e4m3fn cast）

GPU 守卫。H200/L20X（sm90）原生支持 float8 dtype——在真 0.5B 权重上做真实 E4M3 cast，对比 INT8 per-tensor。

In [ ]:
def real_fp8_cast(t, fmt="e4m3"):
    """真 FP8 cast（需 GPU + float8 支持）。"""
    max_v = fp8_max_value(fmt)
    dt = torch.float8_e4m3fn if fmt == "e4m3" else torch.float8_e5m2
    return t.clamp(-max_v, max_v).to(dt).to(torch.float32)

if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, dtype=torch.float16,
                                                 device_map="auto").eval()
    w = model.model.layers[0].mlp.gate_proj.weight.data   # [inter, hidden] fp16 on GPU
    # 真 E4M3 cast
    q_e4m3 = real_fp8_cast(w.float(), "e4m3")
    err_e4m3 = ((q_e4m3 - w.float()).abs().mean()/w.abs().mean()).item()
    # 对照 INT8 per-tensor
    si = 127.0 / w.abs().max()
    q_int8 = torch.round(w*si).clamp(-127,127).to(torch.int8).float()/si
    err_int8 = ((q_int8 - w.float()).abs().mean()/w.abs().mean()).item()
    print(f"真 0.5B gate_proj (GPU 真 cast):")
    print(f"  E4M3 相对误差: {err_e4m3:.4f}")
    print(f"  INT8 per-tensor 相对误差: {err_int8:.4f}")
    print(f"  E4M3 更优: {err_e4m3 < err_int8}")
    # E5M2（范围大但精度粗）
    q_e5m2 = real_fp8_cast(w.float(), "e5m2")
    err_e5m2 = ((q_e5m2 - w.float()).abs().mean()/w.abs().mean()).item()
    print(f"  E5M2 相对误差: {err_e5m2:.4f}（精度粗于 E4M3，正常推理不用）")
    del model; torch.cuda.empty_cache()
else:
    print("跳过 L3：无 GPU（float8 dtype 需 sm89+；CPU 跑 L1/L2 模拟路径）。")

## 产物检查

打印 E4M3 vs E5M2 vs INT8 对比，写 fp8_summary.json。

In [ ]:
def report_fp8():
    torch.manual_seed(13)
    x = torch.randn(1024) * 1.5
    x[20] *= 60.0; x[500] *= 45.0
    summary = {
        "e4m3_max": fp8_max_value("e4m3"),
        "e5m2_max": fp8_max_value("e5m2"),
        "e4m3_relerr": ((quantize_to_fp8(x, "e4m3") - x).abs().mean()/x.abs().mean()).item(),
        "e5m2_relerr": ((quantize_to_fp8(x, "e5m2") - x).abs().mean()/x.abs().mean()).item(),
    }
    si = 127.0/x.abs().max()
    q_int8 = torch.round(x*si).clamp(-127,127).to(torch.int8).float()/si
    summary["int8_per_tensor_relerr"] = ((q_int8 - x).abs().mean()/x.abs().mean()).item()
    (OUT_ROOT/"fp8_summary.json").write_text(json.dumps(summary, indent=2))
    print("== FP8 vs INT8 对比（含 outlier）==")
    for k, v in summary.items():
        print(f"  {k}: {v}")
    print("\n结论：E4M3 误差 < E5M2 < INT8 per-tensor。推理选 E4M3（精度+范围够+抗离群）。")

report_fp8()